# Breast Cancer CNN Classifier - Exploration & Training Notebook

This notebook demonstrates the complete pipeline for IDC detection from histopathology images.

## Table of Contents
1. Environment Setup
2. Data Exploration
3. Data Preprocessing
4. Model Creation
5. Training
6. Evaluation
7. Visualization
8. Model Inference

In [ ]:
# 1. Environment Setup
import sys
import os
sys.path.insert(0, os.path.join('..', 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project imports
from src.utils.config import load_config, config
from src.utils.reproducibility import set_seed, print_system_info, get_device
from src.utils.logging_utils import setup_logging
from src.data import run_full_exploration, create_preprocessor
from src.models import create_model
from src.training import create_trainer
from src.evaluation import evaluate_model, generate_evaluation_plots

# Setup
load_config()
set_seed(config.reproducibility.seeds.python)
device = get_device()
print_system_info()

# Setup logging
logger = setup_logging(level="INFO", log_file="../logs/notebook.log")

In [ ]:
# 2. Data Exploration
print("=" * 60)
print("DATA EXPLORATION")
print("=" * 60)

# Run full exploration
exploration_results = run_full_exploration(
    data_path="../data/raw",
    output_dir="../reports/exploration"
)

# Print summary
summary = exploration_results['summary']
print(f"Total Images: {summary['total_images']}")
print(f"Number of Classes: {summary['num_classes']}")
print(f"Class Distribution: {summary['class_distribution']}")
print(f"Imbalance Ratio: {summary['imbalance_ratio']:.2f}")
print(f"Consistent Dimensions: {summary['consistent_dimensions']}")
print(f"Target Size Match: {summary['target_size_match']}")

In [ ]:
# 3. Data Preprocessing
print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

preprocessor = create_preprocessor()
dataloaders = preprocessor.run_full_pipeline(output_dir="../data/processed")

print(f"Train batches: {len(dataloaders['train'])}")
print(f"Val batches: {len(dataloaders['val'])}")
print(f"Test batches: {len(dataloaders['test'])}")
print(f"Class weights: {preprocessor.class_weights}")

# Inspect a batch
images, labels = next(iter(dataloaders['train']))
print(f"Batch shape: {images.shape}")
print(f"Labels: {labels[:10]}")
print(f"Image range: [{images.min():.4f}, {images.max():.4f}]")

In [ ]:
# 4. Model Creation
print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

# Create custom CNN
model = create_model('CustomCNN')
model.to(device)

# Print model summary
from src.utils.helpers import get_model_summary
print(get_model_summary(model))

In [ ]:
# 5. Training
print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

# Create trainer
trainer = create_trainer(
    model=model,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    class_weights=preprocessor.class_weights,
    device=device
)

# Train (use fewer epochs for demo)
history = trainer.fit(epochs=10)

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
epochs = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train')
axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, history['train_accuracy'], 'b-', label='Train')
axes[0, 1].plot(epochs, history['val_accuracy'], 'r-', label='Val')
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

if 'train_precision' in history:
    axes[1, 0].plot(epochs, history['train_precision'], 'b-', label='Train Precision')
    axes[1, 0].plot(epochs, history['val_precision'], 'r-', label='Val Precision')
    axes[1, 0].plot(epochs, history['train_recall'], 'b--', label='Train Recall')
    axes[1, 0].plot(epochs, history['val_recall'], 'r--', label='Val Recall')
    axes[1, 0].set_title('Precision & Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

if 'lr' in history:
    axes[1, 1].plot(epochs, history['lr'], 'g-')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Evaluation
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Evaluate on test set
results = evaluate_model(
    model=model,
    dataloader=dataloaders['test'],
    device=device,
    class_names=config.data.class_names,
    threshold=config.evaluation.threshold,
    compute_ci=True
)

# Save results
from src.evaluation import ModelEvaluator
evaluator = ModelEvaluator(model, device, config.data.class_names)
evaluator.save_results(results, "../assets/models/evaluation_results.json")
generate_evaluation_plots(results, config.data.class_names, "../assets/models/eval_plots")

In [ ]:
# 7. Visualization
print("=" * 60)
print("VISUALIZATION")
print("=" * 60)

# Confusion Matrix
from src.evaluation import plot_confusion_matrix, plot_roc_curve, plot_metrics_comparison

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_confusion_matrix(results.confusion_matrix, config.data.class_names, ax=axes[0])
plot_roc_curve(results, ax=axes[1])
plot_metrics_comparison(results, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# 8. Model Inference on New Images
print("=" * 60)
print("MODEL INFERENCE")
print("=" * 60)

from src.data import get_val_transforms
from PIL import Image

def predict_image(image_path, model, device, threshold=0.5):
    """Predict class for a single image."""
    transform = get_val_transforms(tuple(config.data.image_size))
    
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        img_array = np.array(img)
    
    augmented = transform(image=img_array)
    img_tensor = augmented['image'].unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        prob_idc = probs[0, 1].item()
        pred_class = 1 if prob_idc >= threshold else 0
    
    return {
        'predicted_class': config.data.class_names[pred_class],
        'idc_probability': prob_idc,
        'threshold': threshold
    }

# Example usage (replace with actual image path)
# result = predict_image("../data/raw/IDC/sample.png", model, device)
# print(f"Prediction: {result['predicted_class']}")
# print(f"IDC Probability: {result['idc_probability']:.4f}")

print("Inference function ready. Use predict_image() with your image paths.")

## Summary

This notebook demonstrated the complete pipeline:

1. **Data Exploration**: Class balance, image dimensions, pixel statistics
2. **Preprocessing**: Stratified splits, augmentation, normalization
3. **Model**: Custom CNN with BatchNorm, Dropout, GlobalAvgPool
4. **Training**: Mixed precision, early stopping, LR scheduling
5. **Evaluation**: Accuracy, Precision, Recall, F1, ROC-AUC, Bootstrap CI
6. **Visualization**: Confusion matrix, ROC curve, metrics comparison
7. **Inference**: Ready-to-use prediction function

For production use, run the full pipeline with `python main.py --mode full` or launch the dashboard with `streamlit run src/dashboard/app.py`.